# Korpus BNPB — versi sederhana

Satu PDF → passage per halaman untuk QA NLP klasik + chunk untuk QA RAG.
Tidak ada training, embedding, indeks, atau pemanggilan LLM pada tahap ini.

**Menjalankan:** install `requirements.txt` dari root repo, pilih kernel Python 3.10+,
lalu **Run all** dari root repo atau folder `corpus/`. Di Colab, upload PDF ke
`knowledge_source/` dan jalankan `%pip install PyMuPDF==1.26.6 transformers==4.57.6 tokenizers==0.22.2 sentencepiece==0.2.3`
bila paket belum tersedia. Tokenizer diunduh pada penggunaan pertama; CPU cukup.

Notebook ini berdiri sendiri. Hasil masuk ke `coba_output/`, terpisah dari hasil notebook lama.
Passage per halaman dipilih agar kondisi, daftar tindakan, dan konteks sebelum/sesudah bencana
masih terbaca bersama. Korpus tetap perlu review manusia, terutama tabel dan infografik.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
from collections import Counter

import pymupdf
from transformers import AutoTokenizer

BASE_DIR = Path.cwd()
if (BASE_DIR / "corpus" / "knowledge_source").is_dir():
    BASE_DIR = BASE_DIR / "corpus"
PDF_PATH = BASE_DIR / "knowledge_source" / "Buku Saku Bencana BNPB.pdf"
OUTPUT_DIR = BASE_DIR / "coba_output"
MAX_TOKENS, OVERLAP_TOKENS = 350, 50
TOKENIZER_ID = "intfloat/multilingual-e5-small"
TOKENIZER_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
PDF_SHA256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
# Koordinat tabel di bawah khusus untuk salinan PDF ini.
assert PDF_SHA256 == "2d96ec6767f06f42a88669e48ab14974042cda5accbbcf3f81e08645dca84f4a", "PDF sumber berbeda. Periksa tata letak dahulu."

/home/fast9/Secondary/CS5/nlp/ta-nlp/venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Pilih halaman isi

Nomor di bawah adalah nomor PDF (mulai 1). Sampul, formulir pribadi, kata pengantar,
daftar isi, halaman pemisah, serta halaman catatan tidak dimasukkan.
Sembilan topik utama dan materi pendukung tetap tercakup. Label bagian adalah metadata,
bukan tambahan fakta dari luar buku. Fase tetap ada di teks halaman; kita tidak menebak fase otomatis.

In [2]:
SECTIONS = [
    (10, 10, 'Ancaman geologi'),
    (11, 12, 'Ancaman hidrometeorologi'),
    (13, 14, 'Penanggulangan bencana dan posko'),
    (16, 19, 'Gempa bumi'),
    (20, 26, 'Tsunami'),
    (27, 33, 'Erupsi gunungapi'),
    (34, 41, 'Banjir'),
    (42, 46, 'Tanah longsor'),
    (47, 50, 'Puting beliung'),
    (51, 54, 'Kekeringan'),
    (55, 58, 'Kebakaran hutan dan lahan'),
    (59, 62, 'Pandemi COVID-19'),
    (64, 68, 'Kesiapsiagaan umum'),
    (69, 69, 'Mitigasi untuk penyandang disabilitas'),
    (71, 75, 'Mitigasi vegetasi'),
    (76, 76, 'Kesiapsiagaan hewan peliharaan'),
    (77, 80, 'Klasifikasi bahan berbahaya dan beracun'),
    (82, 82, 'Rambu dan papan informasi bencana'),
    (83, 83, 'Aplikasi gawai dalam buku'),
    (84, 91, 'Kontak dan BPBD dalam buku'),
    (92, 92, 'Daftar singkatan'),
]

## 2. Ekstraksi teks dengan sedikit aturan tata letak

Pembersihan hanya merapikan Unicode dan spasi. Huruf besar, tanda baca, angka, satuan,
serta kata negasi dipertahankan dalam `text`. Footer dibuang menurut posisi dan teksnya.
`search_text` dan `tokens` dibuat terpisah untuk regex/TF-IDF/BM25; stemming dan stopword
removal dapat diuji nanti tanpa mengubah bukti asli atau membuang kata seperti **tidak/jangan**.

Ekstraksi umum membaca blok dari atas ke bawah. Pengecualian ringkas di bawah menjaga
pasangan kolom tabel, kolom disabilitas, dan label/deskripsi tas siaga serta hewan peliharaan.
Koordinatnya berasal dari pemeriksaan PDF; tidak ada jawaban Q&A yang disisipkan.
Teks gambar yang tidak mempunyai lapisan teks dan hubungan garis diagram tidak ditafsirkan otomatis.

In [3]:
TABLE_ROWS = {
    12: [(230, 264), (267, 324), (329, 358)],
    29: [(181, 228), (240, 295), (305, 342), (355, 392)],
    84: [(45, 85), (86, 103), (104, 130), (131, 147), (148, 164), (165, 181), (182, 197), (198, 224), (225, 241), (242, 258), (259, 274), (275, 291), (292, 308), (309, 334), (335, 360), (361, 380)],
    85: [(95, 184), (185, 238), (239, 292), (294, 339), (340, 390)],
    86: [(33, 75), (75, 127), (127, 161), (161, 211), (211, 294), (294, 367)],
    87: [(79, 167), (167, 232), (232, 329), (329, 394)],
    88: [(33, 110), (110, 187), (187, 229), (229, 303), (303, 390)],
    89: [(79, 155), (155, 228), (228, 281), (281, 325), (325, 390)],
    90: [(31, 86), (86, 138), (138, 183), (183, 236), (236, 288), (288, 390)],
    91: [(79, 120), (120, 173), (173, 390)],
}
TABLE_COLUMNS = {
    12: [(25, 175), (178, 270)], 29: [(15, 54), (54, 94), (94, 272)],
    84: [(47, 175), (175, 278)],
}
# Region dibaca berurutan. Region tabel di atas ditambahkan sesudah pengantar.
REGIONS = {
    12: [(165, 65, 270, 112), (20, 145, 270, 210)],
    29: [(20, 40, 270, 78)],
    30: [(20, 13, 270, 32), (90, 76, 273, 120), (90, 124, 273, 235), (90, 239, 273, 377)],
    69: [(20, 75, 275, 100), (49, 102, 145, 132), (49, 138, 140, 167), (192, 102, 275, 132),
         (4, 258, 102, 399), (104, 274, 202, 399), (203, 300, 278, 399)],
}
REGIONS[67] = [[30, 89, 265, 151], [10, 194, 98, 234], [10, 233, 65, 293], [10, 295, 70, 314], [10, 314, 85, 331], [10, 333, 99, 369], [10, 369, 99, 398], [211, 194, 275, 239], [215, 240, 275, 292], [212, 293, 275, 330], [208, 331, 275, 368], [188, 369, 275, 399], [100, 361, 185, 400]]
REGIONS[76] = [[15, 57, 278, 160], [27, 185, 160, 218], [25, 219, 135, 256], [25, 266, 140, 306], [177, 170, 278, 261], [214, 265, 270, 306], [60, 323, 150, 338], [60, 338, 150, 357], [60, 357, 151, 383], [187, 319, 275, 346], [187, 346, 275, 363], [187, 365, 275, 379]]

REGIONS[65] = [[65, 42, 273, 79], [60, 92, 275, 117], [60, 148, 275, 176], [60, 193, 275, 231], [60, 244, 275, 281], [60, 303, 275, 322], [60, 353, 275, 373]]

# Koreksi transkripsi yang dicocokkan dengan tampilan PDF, bukan koreksi fakta.
TEXT_FIXES = {
    17: [("bumi,\n\nwww.bmkg.go.id\n\nkedalaman", "bumi, kedalaman"), ("www.bmkg. go.id", "www.bmkg.go.id")],
    25: [("\n\nr\n\n", "\n\n")],
    26: [("T\n\netap", "Tetap")],
    35: [("banji\n\nr.", "banjir.")],
    41: [("T\n\nerlibat", "Terlibat")],
    64: [("26 APRIL sebagai Hari Kesiapsiagaan Bencana BNPB menetapkan", "BNPB menetapkan 26 APRIL sebagai Hari Kesiapsiagaan Bencana")],
}


def clean(text):
    text = unicodedata.normalize("NFKC", text).replace("\u00ad", "")
    return re.sub(r"\s+", " ", text).strip()


def blocks_in(page, rect):
    blocks = page.get_text("blocks", clip=pymupdf.Rect(rect))
    return [clean(b[4]) for b in sorted(blocks, key=lambda b: (round(b[1]), b[0]))
            if b[6] == 0 and clean(b[4])]


def extract_page(page, number):
    if number in TABLE_ROWS:
        paragraphs = [clean(page.get_text(clip=pymupdf.Rect(r))) for r in REGIONS.get(number, [])]
        columns = TABLE_COLUMNS.get(number, [(47, 138), (138, 278)])
        for top, bottom in TABLE_ROWS[number]:
            parts = [clean(page.get_text(clip=pymupdf.Rect(left, top, right, bottom)))
                     for left, right in columns]
            paragraphs.append(" | ".join(parts))
    elif number in REGIONS:
        paragraphs = [clean(page.get_text(clip=pymupdf.Rect(r))) for r in REGIONS[number]]
    else:
        paragraphs = blocks_in(page, (0, 0, page.rect.width, 415))
    paragraphs = [s for s in paragraphs if s and not re.fullmatch(r"[\d\s.]+", s)
                  and "TANGGAP TANGKAS TANGGUH" not in s and s != "MENGHADAPI BENCANA"]
    return "\n\n".join(paragraphs)


def normalize_search(text):
    return clean(text).casefold()


def tokenize(text):
    return re.findall(r"\w+(?:[-.,/]\w+)*", normalize_search(text))


def page_flags(number):
    flags = ["pending_review"]
    if number in REGIONS or number in TABLE_ROWS or number in {14, 56, 77, 78, 79, 80, 82, 83}:
        flags.append("layout_review")
    if 59 <= number <= 62 or 83 <= number <= 91:
        flags.append("historical_information")
    if number in {28, 45, 56, 57, 62} or 71 <= number <= 75:
        flags.append("source_claim_review")
    return flags


passages = []
with pymupdf.open(PDF_PATH) as pdf:
    assert len(pdf) == 98
    for first, last, section in SECTIONS:
        for number in range(first, last + 1):
            text = extract_page(pdf[number - 1], number)
            fixes = TEXT_FIXES.get(number, [])
            for before, after in fixes:
                assert text.count(before) == 1, f"Periksa ulang koreksi transkripsi PDF {number}"
                text = text.replace(before, after)
            passages.append({
                "id": f"bnpb-{PDF_SHA256[:8]}-p{number:03}", "section": section,
                "text": text, "retrieval_text": section + "\n" + text,
                "search_text": normalize_search(section + " " + text), "tokens": tokenize(section + " " + text),
                "source": {"file": PDF_PATH.name, "pdf_page": number, "book_page": number - 2},
                "flags": page_flags(number), "transcription_fixes": fixes,
            })
print(f"{len(passages)} passage halaman | {len(SECTIONS)} bagian")

79 passage halaman | 21 bagian


## 3. Chunk RAG dengan rujukan ke passage

Gunakan tokenizer model embedding yang akan dipakai, bukan perkiraan jumlah kata.
Di sini tokenizer E5 dipin versinya; tidak memuat bobot model. Batas 350 menghitung
`passage: ` + judul bagian + isi + special tokens. Chunk tidak melintasi halaman.
Jika halaman panjang, batas paragraf diprioritaskan; fallback memakai batas token,
dengan overlap sekitar 50 token untuk menjaga konteks di batas chunk.

`start_char` dan `end_char` menunjuk ke `text` passage induk sehingga chunk dan jawaban
bisa diaudit. Offset berlaku pada teks hasil ekstraksi, bukan koordinat PDF.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID, revision=TOKENIZER_REVISION, use_fast=True, trust_remote_code=False
)
assert 0 <= OVERLAP_TOKENS < MAX_TOKENS <= tokenizer.model_max_length


def embedding_input(section, text):
    return "passage: " + section + "\n" + text


def count_tokens(section, text):
    return len(tokenizer.encode(embedding_input(section, text), add_special_tokens=True))


chunks = []
for passage in passages:
    text, section = passage["text"], passage["section"]
    offsets = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    budget = MAX_TOKENS - count_tokens(section, "") - 8  # margin batas token prefix/isi
    assert budget > OVERLAP_TOKENS
    start = 0
    while start < len(text):
        remaining = [(a, b) for a, b in offsets if b > start]
        end = remaining[min(budget, len(remaining)) - 1][1]
        if len(remaining) <= budget:
            end = len(text)
        else:
            paragraph_end = text.rfind("\n\n", start, end)
            if paragraph_end > start + (end - start) // 2:
                end = paragraph_end + 2
        while count_tokens(section, text[start:end]) > MAX_TOKENS:
            end = max(b for a, b in offsets if start < b < end)
        chunk_text = text[start:end]
        chunks.append({
            "id": f"{passage['id']}-c{start:04}", "passage_id": passage["id"],
            "section": section, "text": chunk_text, "retrieval_text": section + "\n" + chunk_text,
            "start_char": start, "end_char": end, "token_count": count_tokens(section, chunk_text),
            "source": passage["source"], "flags": passage["flags"],
        })
        if end == len(text):
            break
        covered = [a for a, b in offsets if start < a < end]
        start = covered[max(0, len(covered) - OVERLAP_TOKENS)] if covered and OVERLAP_TOKENS else end
print(f"{len(chunks)} chunk | token terbesar: {max(c['token_count'] for c in chunks)}")

80 chunk | token terbesar: 330


## 4. Pemeriksaan struktur dan simpan

Pemeriksaan ini menguji isi tidak kosong, rujukan halaman, cakupan teks setelah chunking,
dan batas token. Ini **belum mengukur kualitas jawaban**. Buka `corpus.md` dan cocokkan
ke PDF, terutama halaman yang memiliki `layout_review` atau `source_claim_review`.

Contoh temuan inspeksi: PDF 28 memuat klaim suhu/jarak yang perlu ditelaah;
PDF 62 butir 11 berakhir pada kata “atau” pada sumber; PDF 25–26/41 memiliki glyph
terpisah. Artefak glyph yang sudah diperiksa diperbaiki lewat `TEXT_FIXES`, dan pasangan sebelum/sesudah dicatat pada passage. Jangan memperbaiki
fakta sumber secara diam-diam. Koreksi transkripsi harus dicatat sebelum korpus dibekukan.

In [5]:
by_id = {p["id"]: p for p in passages}
assert len(by_id) == len(passages) and all(p["text"].strip() for p in passages)
assert len({c["id"] for c in chunks}) == len(chunks)
for passage in passages:
    page_chunks = [c for c in chunks if c["passage_id"] == passage["id"]]
    covered_until = 0
    for c in page_chunks:
        assert c["source"] == passage["source"]
        assert c["text"] == passage["text"][c["start_char"]:c["end_char"]]
        assert 0 < c["token_count"] <= MAX_TOKENS
        assert c["start_char"] <= covered_until < c["end_char"]
        covered_until = c["end_char"]
    assert covered_until == len(passage["text"])

manifest = {
    "pdf_sha256": PDF_SHA256, "tokenizer": TOKENIZER_ID, "revision": TOKENIZER_REVISION,
    "max_tokens": MAX_TOKENS, "overlap_tokens": OVERLAP_TOKENS,
    "passages": len(passages), "chunks": len(chunks), "review_status": "pending_review",
    "flags": dict(Counter(flag for p in passages for flag in p["flags"])),
}
# Identitas ini berubah jika teks, chunk, atau pengaturan berubah; simpan bersama hasil evaluasi.
corpus_hash = hashlib.sha256(json.dumps([manifest, passages, chunks], ensure_ascii=False, sort_keys=True).encode()).hexdigest()
manifest["corpus_version"] = "bnpb-simple-" + corpus_hash[:16]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, rows in [("classic_passages.jsonl", passages), ("rag_chunks.jsonl", chunks)]:
    (OUTPUT_DIR / name).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
review = ["# Korpus BNPB sederhana", manifest["corpus_version"], "Status: pending_review"]
for p in passages:
    review += [f"## {p['section']} — PDF {p['source']['pdf_page']} / buku {p['source']['book_page']}",
               f"ID: {p['id']} | Flags: {', '.join(p['flags'])}", p["text"]]
(OUTPUT_DIR / "corpus.md").write_text("\n\n".join(review), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Output:", OUTPUT_DIR.resolve())

{
  "pdf_sha256": "2d96ec6767f06f42a88669e48ab14974042cda5accbbcf3f81e08645dca84f4a",
  "tokenizer": "intfloat/multilingual-e5-small",
  "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
  "max_tokens": 350,
  "overlap_tokens": 50,
  "passages": 79,
  "chunks": 80,
  "review_status": "pending_review",
  "flags": {
    "pending_review": 79,
    "layout_review": 23,
    "source_claim_review": 10,
    "historical_information": 13
  },
  "corpus_version": "bnpb-simple-3b9bdbec7fd0af9a"
}
Output: /home/fast9/Secondary/CS5/nlp/ta-nlp/corpus/coba_output


## 5. Pemakaian dan evaluasi dengan Q&A pairs

**NLP klasik:** indeks `retrieval_text`/`tokens` dari `classic_passages.jsonl` dengan
TF-IDF atau BM25. Cari passage dahulu, lalu jalankan regex atau ekstraksi kalimat
pada `text`. Normalisasi pertanyaan dengan fungsi yang sama. `tokens` di sini hanya
tokenisasi regex dasar, bukan analisis morfologi. Simpan jawaban dalam ejaan aslinya.

**RAG:** indeks `rag_chunks.jsonl`. Jika memakai E5, embedding dokumen memakai
`"passage: " + row["retrieval_text"]` dan embedding pertanyaan memakai `"query: " + question`.
Kirim teks chunk dan `source` ke LLM. `token_count` khusus tokenizer embedding E5;
budget prompt LLM dihitung lagi memakai tokenizer LLM. Prefix E5 tidak ditambahkan dua kali.

**Q&A evaluasi disimpan terpisah**, misalnya `evaluation/qa_pairs.jsonl` dengan field:
`id`, `question`, `answers` (daftar variasi jawaban benar), `source_pdf_pages` (daftar
halaman bukti), `split` (`dev` atau `test`), dan `answerable` (boolean).
Jawaban dibuat dan diperiksa manusia terhadap PDF. Jangan memasukkan pertanyaan/gold answer
ke indeks korpus. Pertanyaan parafrasa atau dari fakta yang sama harus berada di split yang sama.

1. Review korpus dahulu, kemudian bekukan `corpus_version` dan Q&A. Gunakan pasangan Q&A yang sama untuk kedua sistem.
2. Atur regex, top-k, chunking, dan prompt hanya memakai split **dev**. Jalankan **test** setelah pengaturan dikunci.
3. Ukur retrieval dengan **Recall@k pada halaman bukti** agar passage halaman dan chunk bisa dibandingkan.
   Untuk pertanyaan multi-halaman, laporkan proporsi halaman bukti yang ditemukan, bukan sekadar satu halaman cocok.
4. Ukur jawaban dengan **Exact Match** terhadap variasi gold dan **token F1** (hitung kemunculan token).
   Pakai normalisasi evaluasi yang sama; pertahankan angka dan negasi. Laporkan juga review makna untuk jawaban LLM yang berupa parafrasa.
5. Sertakan pertanyaan yang jawabannya tidak tersedia. Ukur kemampuan sistem menolak menjawab,
   serta ketepatan kutipan halaman dan dukungan bukti untuk jawaban LLM.

Korpus ini belum merupakan sistem QA dan belum menjamin skor evaluasi tertentu.
Teks historis dinilai sebagai isi buku bila dimasukkan ke set evaluasi; pertanyaan tentang
informasi terkini memerlukan sumber lain yang memang dipelihara.